# Sección 0 · Introducción: la lenga y sus datos

*Contexto biológico y origen de los datos, antes de tocar código*

> **Esta sección casi no tiene código:** es el contexto para entender qué vamos a analizar y de dónde salen los números. Al final hay un mapa de los sitios de muestreo y una tarea breve.


## Objetivos

Al terminar esta sección sabrás:

1. Qué es la **lenga** (*Nothofagus pumilio*) y por qué estudiamos su genética.
2. De dónde vienen los datos: el **estudio de referencia** y su **diseño de muestreo**.
3. Cómo se **genera un VCF**: el camino de la hoja al archivo de variantes.
4. Qué datos usaremos en el curso (simulados vs. reales).


## 1. La especie: *Nothofagus pumilio* (lenga)

La **lenga** es el árbol dominante de los bosques subantárticos de Sudamérica. Crece a lo largo de un gradiente de casi **2.000 km** en los Andes del sur (≈36°–55°S), desde el norte de la Patagonia argentino-chilena hasta Tierra del Fuego.

- Es una **especie fundadora** (*foundation species*): estructura el ecosistema donde vive.
- Tiene valor **económico** (madera) y **ecológico** (hábitat, captura de carbono, cuencas).
- Tolera condiciones extremas: viento, frío y una estación de crecimiento corta.

> La pregunta que nos guía: **¿cómo se distribuye la variación genética de la lenga a lo largo de ese gradiente, y cuánta de esa variación está asociada al ambiente?**


## 2. ¿Por qué estudiar su genética?

La distribución de la lenga cruza **dos gradientes ambientales casi perpendiculares**:

- **Norte → Sur:** temperatura y fotoperiodo (más frío y días más cortos hacia el sur).
- **Oeste → Este:** precipitación (húmedo hacia la costa, seco hacia la estepa).

Además, la **altitud** añade un gradiente local de temperatura en cada sitio.

**¿Por qué importa?** Si las poblaciones están **adaptadas localmente** a su clima, el cambio climático puede desacoplarlas de su ambiente. Entender esa adaptación sirve para predecir riesgos y priorizar poblaciones para la **conservación**.


## 3. El estudio de referencia (Sekely et al. 2024)

Nuestros datos reales vienen del estudio de **Sekely et al. (2024)**, *Plants, People, Planet*: *Genomic responses to climate: understanding local adaptation in the Andean tree species* Nothofagus pumilio*.

- **20 sitios** de muestreo a lo largo del rango de la especie, la mayoría con **pares de elevación** (alto/bajo).
- **502 árboles** (496 adultos + 6 réplicas).
- **Secuenciación por enriquecimiento de sondas** (*target enrichment*): se capturan regiones específicas del genoma (genes candidatos y regiones asociadas a clima), no el genoma completo.
- Tras el filtrado: ~**47.000 SNP** repartidos en ~1.600 contigs.
- Hallazgo principal: la **estructura genética se alinea con la latitud**, con señales de adaptación local al clima.

> Paper (acceso abierto): <https://doi.org/10.1002/ppp3.10504> · Datos (Zenodo, CC-BY-4.0): <https://doi.org/10.5281/zenodo.10018374>


## 4. Cómo se genera un VCF: de la hoja al archivo

Un VCF no aparece de la nada: es el final de una **tubería bioinformática** (*pipeline*). El camino completo:

```
1. Muestreo      → se colecta tejido (hojas) de cada árbol, anotando sitio y coordenadas.
2. Extracción    → se aísla el ADN de cada muestra.
3. Secuenciación → un secuenciador (Illumina) lee fragmentos cortos de ADN (reads).
4. Mapeo         → cada read se alinea a un genoma de referencia (aquí, un ensamblaje de lenga).
5. Llamado       → el software (GATK HaplotypeCaller) compara los reads contra la referencia
                   y detecta variantes: posiciones donde una muestra difiere de la referencia.
6. Filtrado      → se descartan variantes de baja calidad (poca cobertura, sesgo de hebra…).
7. VCF           → el resultado es un archivo VCF: una variante por fila, un genotipo por muestra.
```

**¿Qué es una variante (SNP)?** Una posición del genoma donde individuos distintos llevan nucleótidos distintos. El genotipo de cada individuo se codifica como `0/0`, `0/1`, `1/1` o `./.` (faltante).

**¿Por qué «contigs» y no cromosomas?** Como se usó enriquecimiento por sondas, el ensamblaje de referencia está fragmentado en **contigs** (pedazos con nombres como `NODE_12345`), no en cromosomas completos. Lo veremos al mirar el VCF.

> La explicación de cada columna del VCF está en [`docs/formato_vcf.md`](docs/formato_vcf.md).


## 5. Los datos del curso: dos fuentes

Usaremos **dos tipos de datos**:

| | Simulados | Reales (subconjunto) |
|---|---|---|
| Origen | `00_simular_datos.R` | Sekely et al. 2024 (Zenodo, CC-BY-4.0) |
| Tamaño | 20 poblaciones × 12 árboles × 1.500 SNP | 3 sitios × 6 árboles × 301 SNP |
| Para qué | Practicar con una «verdad» conocida | Ver cómo se ve un VCF real |

- Los **simulados** imitan a la lenga (gradiente latitudinal, más deriva al sur) e incluyen trampas para entrenar el ojo (una muestra mal etiquetada, datos faltantes).
- Los **reales** son un recorte diminuto y limpio para empezar: `datos_ejemplo/lenga_ejemplo.vcf`.


## 6. Mapa de los 20 sitios

Veamos dónde están los sitios de muestreo del estudio real:


In [ ]:
# ---- Preparación ----
options(repr.plot.width = 7, repr.plot.height = 5, repr.plot.res = 110)
library(ggplot2)

sitios <- read.csv("datos_ejemplo/lenga_sitios.csv", stringsAsFactors = FALSE)

ggplot(sitios, aes(lon, lat, colour = el_class, size = elev)) +
  geom_point(alpha = 0.85) +
  geom_text(aes(label = loc), nudge_y = 0.35, size = 3, colour = "grey30", show.legend = FALSE) +
  scale_colour_viridis_d(option = "D") +
  scale_size_continuous(range = c(2, 6)) +
  labs(title = "20 sitios de muestreo (Sekely et al. 2024)",
       x = "Longitud", y = "Latitud",
       colour = "Clase de elevación", size = "Elevación (m)") +
  coord_fixed(1.3)


Fíjate en el eje de latitud: abarca casi 20° (~2.000 km). Los sitios **1** (Epulaufquen), **10** (Lago Guacho) y **20** (Ushuaia) son los tres que usamos en el subconjunto real del curso.


## Resumen: ideas que debes llevarte

1. La lenga es una especie fundadora que cruza gradientes fuertes de **clima** y **fotoperiodo**.
2. El estudio de referencia (Sekely et al. 2024) muestreó **20 sitios** con pares de elevación.
3. Un VCF es el resultado de un **pipeline**: muestreo → secuenciación → mapeo → llamado → filtrado.
4. En el curso usamos **datos simulados** (para practicar) y un **subconjunto real** (para ver datos de verdad).


## Tarea

1. Lee el **abstract** del paper de Sekely et al. (2024) (enlace en la sección 3) y escribe en 2 líneas cuál fue su pregunta y su hallazgo principal.
2. Con el mapa: formula una **hipótesis** sobre dónde esperas más diversidad genética (norte vs. sur) y por qué. La retomaremos en la Sección 2.
